# 01. Data Profiling — Indonesian Religious Corpus

## Tujuan

Notebook ini merupakan bagian dari **Milestone 1 Tugas 1 Analisis Big Data 2026**.

Tujuan utama profiling adalah memahami karakteristik awal dataset sebelum dilakukan proses cleaning dan analisis eksploratif lebih lanjut.

Profiling mencakup:

- verifikasi ukuran file dan jumlah data;
- jumlah baris dan kolom;
- struktur serta tipe data;
- missing values;
- duplikasi;
- statistik dasar;
- distribusi variabel kategorikal;
- karakteristik variabel teks;
- pemeriksaan awal kolom tanggal dan lokasi apabila tersedia.

> **Catatan:** Notebook ini berfokus pada **profiling awal**. Proses cleaning, penanganan outlier, aturan kualitas data, dan profiling DuckDB `SUMMARIZE` dikerjakan pada **Milestone 2**.


## 1. Import Library

Library utama yang digunakan adalah **Polars** karena dataset berukuran besar dan tugas mewajibkan penggunaan Polars untuk data handling.

Pendekatan **lazy evaluation** digunakan untuk operasi yang memungkinkan, sehingga proses transformasi dan agregasi dapat direncanakan terlebih dahulu sebelum dieksekusi.


In [1]:
import os
from pathlib import Path

import polars as pl

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)

print("Polars version:", pl.__version__)


Polars version: 1.44.2


## 2. Konfigurasi Dataset

File dataset berada pada folder:

```text
data/raw/data_indonesian_religius.csv
```

Karena notebook berada di dalam folder `notebooks/`, path relatif yang digunakan adalah:

```text
../data/raw/data_indonesian_religius.csv
```


In [2]:
DATA_PATH = Path("../data/raw/data_indonesian_religius.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATA_PATH.resolve()}\n"
        "Pastikan file berada di folder data/raw/"
    )

print("Dataset:", DATA_PATH.resolve())


Dataset: /home/jovyan/work/data/raw/data_indonesian_religius.csv


## 3. Verifikasi Ukuran File

Syarat utama Milestone 1 adalah dataset memiliki ukuran minimal **500 MB atau lebih dari 1 juta baris**.

Pada bagian ini ukuran file diperiksa langsung dari file lokal yang digunakan dalam project, sehingga bukti yang ditampilkan berasal dari dataset yang benar-benar dianalisis.


In [3]:
file_size_bytes = DATA_PATH.stat().st_size
file_size_mb = file_size_bytes / (1024 ** 2)
file_size_gb = file_size_bytes / (1024 ** 3)

print(f"Ukuran file : {file_size_mb:,.2f} MB")
print(f"             {file_size_gb:,.2f} GB")


Ukuran file : 682.73 MB
             0.67 GB


## 4. Membaca Dataset dengan Polars Lazy API

Dataset dibaca menggunakan `pl.scan_csv()` sehingga Polars menggunakan **lazy execution**.

Berbeda dari pembacaan eager, `scan_csv()` tidak langsung menjalankan seluruh pipeline. Operasi dapat disusun terlebih dahulu dan baru dieksekusi ketika `.collect()` dipanggil.

`infer_schema_length` digunakan untuk membaca sejumlah baris awal sebagai dasar inferensi tipe data.


In [4]:
lf = pl.scan_csv(
    DATA_PATH,
    infer_schema_length=10_000,
    try_parse_dates=False,
    null_values=["", "NULL", "null", "None"],
)

print("LazyFrame berhasil dibuat.")
print(lf)


LazyFrame berhasil dibuat.
naive plan: (run LazyFrame.explain(optimized=True) to see the optimized plan)

Csv SCAN [../data/raw/data_indonesian_religius.csv]
PROJECT */7 COLUMNS
ESTIMATED ROWS: 4312615


## 5. Struktur dan Nama Kolom

Bagian ini digunakan untuk mengetahui nama kolom serta tipe data yang dikenali Polars.

Karena struktur kolom dataset digunakan pada analisis berikutnya, nama kolom tidak ditulis secara hard-code. Notebook mengambilnya langsung dari schema dataset.


In [5]:
schema = lf.collect_schema()

print("Jumlah kolom:", len(schema))
print("\nSchema dataset:")

for column, dtype in schema.items():
    print(f"- {column}: {dtype}")


Jumlah kolom: 7

Schema dataset:
- Label: String
- Denomination: String
- Location: String
- Date: String
- Title: String
- Sentence_Unit: String
- Link: String


## 6. Jumlah Baris dan Kolom

Jumlah baris dihitung menggunakan operasi lazy Polars. Informasi ini menjadi salah satu bukti bahwa dataset memenuhi persyaratan skala Milestone 1.


In [6]:
shape_profile = (
    lf
    .select([
        pl.len().alias("jumlah_baris")
    ])
    .collect()
)

row_count = shape_profile["jumlah_baris"][0]
column_count = len(schema)

print(f"Jumlah baris  : {row_count:,}")
print(f"Jumlah kolom  : {column_count:,}")

meets_size_requirement = file_size_mb >= 500
meets_row_requirement = row_count > 1_000_000

print("\nPemeriksaan syarat Milestone 1:")
print(f"- Ukuran >= 500 MB   : {meets_size_requirement}")
print(f"- Baris > 1 juta     : {meets_row_requirement}")
print(f"- Syarat terpenuhi   : {meets_size_requirement or meets_row_requirement}")


Jumlah baris  : 3,013,172
Jumlah kolom  : 7

Pemeriksaan syarat Milestone 1:
- Ukuran >= 500 MB   : True
- Baris > 1 juta     : True
- Syarat terpenuhi   : True


## 7. Profil Tipe Data

Tipe data setiap kolom ditampilkan dalam bentuk tabel agar struktur dataset lebih mudah diperiksa.


In [7]:
dtype_profile = pl.DataFrame({
    "column": list(schema.keys()),
    "dtype": [str(dtype) for dtype in schema.values()],
})

dtype_profile


column,dtype
str,str
"""Label""","""String"""
"""Denomination""","""String"""
"""Location""","""String"""
"""Date""","""String"""
"""Title""","""String"""
"""Sentence_Unit""","""String"""
"""Link""","""String"""


## 8. Missing Values

Missing values dihitung untuk setiap kolom.

Pada tahap Milestone 1, missing values **hanya diidentifikasi dan dideskripsikan**. Belum dilakukan penghapusan atau imputasi karena penanganan missing values merupakan bagian dari Milestone 2.


In [8]:
missing_profile = (
    lf
    .select([
        pl.col(column).null_count().alias(column)
        for column in schema.keys()
    ])
    .collect()
)

missing_profile


Label,Denomination,Location,Date,Title,Sentence_Unit,Link
u32,u32,u32,u32,u32,u32,u32
0,0,0,3013172,3013172,0,0


In [9]:
missing_summary = (
    missing_profile
    .transpose(
        include_header=True,
        header_name="column",
        column_names=["missing_count"]
    )
    .with_columns([
        pl.col("missing_count").cast(pl.Int64),
        pl.lit(row_count).alias("total_rows"),
    ])
    .with_columns(
        (pl.col("missing_count") / pl.col("total_rows") * 100)
        .round(4)
        .alias("missing_percentage")
    )
    .sort("missing_count", descending=True)
)

missing_summary


column,missing_count,total_rows,missing_percentage
str,i64,i32,f64
"""Date""",3013172,3013172,100.0
"""Title""",3013172,3013172,100.0
"""Label""",0,3013172,0.0
"""Denomination""",0,3013172,0.0
"""Location""",0,3013172,0.0
"""Sentence_Unit""",0,3013172,0.0
"""Link""",0,3013172,0.0


## 9. Pemeriksaan Duplikasi

Jumlah baris duplikat diperiksa sebagai bagian dari data profiling.

Pemeriksaan ini belum melakukan penghapusan data. Jika ditemukan duplikasi, keputusan penanganannya akan dibahas pada notebook cleaning di Milestone 2.


In [11]:
# ============================================================
# PROFILING DUPLIKAT
# ============================================================

unique_rows = (
    lf.unique()
    .select(pl.len().alias("unique_rows"))
    .collect()
    .item()
)

duplicate_rows = row_count - unique_rows

duplicate_percentage = (
    (duplicate_rows / row_count) * 100
    if row_count > 0
    else 0.0
)

duplicate_profile = pl.DataFrame(
    {
        "metric": [
            "total_rows",
            "unique_rows",
            "duplicate_rows",
            "duplicate_percentage",
        ],
        "value": [
            str(row_count),
            str(unique_rows),
            str(duplicate_rows),
            f"{duplicate_percentage:.6f}",
        ],
    }
)

display(duplicate_profile)

metric,value
str,str
"""total_rows""","""3013172"""
"""unique_rows""","""3012817"""
"""duplicate_rows""","""355"""
"""duplicate_percentage""","""0.011782"""


## 10. Statistik Dasar Numerik

Kolom numerik diidentifikasi secara otomatis. Jika terdapat kolom numerik, statistik dasar seperti jumlah data, rata-rata, standar deviasi, minimum, median, dan maksimum dihitung.

Jika dataset tidak memiliki kolom numerik, bagian ini tetap berjalan tanpa menghasilkan error.


In [12]:
numeric_columns = [
    column
    for column, dtype in schema.items()
    if dtype.is_numeric()
]

print("Kolom numerik:", numeric_columns)

if numeric_columns:
    numeric_profile = (
        lf
        .select([
            pl.col(column).cast(pl.Float64).mean().alias(f"{column}_mean")
            for column in numeric_columns
        ])
        .collect()
    )

    numeric_describe = (
        lf
        .select(numeric_columns)
        .collect()
        .describe()
    )

    numeric_describe
else:
    print("Tidak terdapat kolom numerik pada dataset.")


Kolom numerik: []
Tidak terdapat kolom numerik pada dataset.


## 11. Profil Kolom Kategorikal

Kolom dengan tipe `String`, `Categorical`, atau `Enum` diperiksa untuk mengetahui jumlah kategori unik dan kategori yang paling sering muncul.

Analisis ini merupakan **profiling**, bukan keputusan cleaning.


In [13]:
categorical_columns = [
    column
    for column, dtype in schema.items()
    if dtype in (pl.String, pl.Categorical, pl.Enum)
]

print("Kolom kategorikal/teks:", categorical_columns)


Kolom kategorikal/teks: ['Label', 'Denomination', 'Location', 'Date', 'Title', 'Sentence_Unit', 'Link']


In [17]:
categorical_summary_rows = []

for column in categorical_columns:
    result = (
        lf
        .select([
            pl.col(column).n_unique().alias("unique_values"),
            pl.col(column).null_count().alias("missing_values"),
        ])
        .collect()
    )

    categorical_summary_rows.append({
        "column": column,
        "unique_values": result["unique_values"][0],
        "missing_values": result["missing_values"][0],
    })

if categorical_summary_rows:
    categorical_summary = (
        pl.DataFrame(categorical_summary_rows)
        .sort("unique_values", descending=True)
    )
    categorical_summary
else:
    print("Tidak terdapat kolom kategorikal.")


## 12. Distribusi Nilai Kategorikal Utama

Untuk kolom kategorikal dengan jumlah kategori yang relatif terbatas, distribusi nilai ditampilkan sebagai frekuensi dan persentase.

Kolom dengan terlalu banyak nilai unik tidak langsung ditampilkan seluruhnya agar profiling tetap efisien.


In [18]:
MAX_CATEGORIES_TO_DISPLAY = 30

for column in categorical_columns:
    unique_count = (
        lf
        .select(pl.col(column).n_unique().alias("n"))
        .collect()["n"][0]
    )

    if unique_count <= MAX_CATEGORIES_TO_DISPLAY:
        print(f"\n=== {column} ({unique_count:,} kategori) ===")

        distribution = (
            lf
            .group_by(column, maintain_order=True)
            .agg(pl.len().alias("count"))
            .with_columns(
                (pl.col("count") / row_count * 100)
                .round(4)
                .alias("percentage")
            )
            .sort("count", descending=True)
            .collect()
        )

        display(distribution)
    else:
        print(
            f"\n{column}: {unique_count:,} kategori unik "
            f"(distribusi lengkap tidak ditampilkan)."
        )



=== Label (3 kategori) ===


Label,count,percentage
str,u32,f64
"""Islam""",1455454,48.3031
"""Catholic""",797131,26.4549
"""Protestant""",760587,25.2421



Denomination: 60 kategori unik (distribusi lengkap tidak ditampilkan).

=== Location (9 kategori) ===


Location,count,percentage
str,u32,f64
"""National""",2492876,82.7326
"""Java""",286452,9.5067
"""Sumatra""",144632,4.8
"""Sulawesi""",30024,0.9964
"""Nusa Tenggara""",23684,0.786
"""Papua""",19199,0.6372
"""International""",9369,0.3109
"""Maluku""",5988,0.1987
"""Kalimantan""",948,0.0315



=== Date (1 kategori) ===


Date,count,percentage
str,u32,f64
null,3013172,100.0



=== Title (1 kategori) ===


Title,count,percentage
str,u32,f64
null,3013172,100.0



Sentence_Unit: 1,966,151 kategori unik (distribusi lengkap tidak ditampilkan).

Link: 132,075 kategori unik (distribusi lengkap tidak ditampilkan).


## 13. Profil Kolom Teks

Dataset ini merupakan corpus teks sehingga karakteristik teks menjadi bagian penting dari profiling.

Jika kolom berikut tersedia:

- `Sentence_Unit`
- `Title`

akan dihitung:

- jumlah karakter;
- jumlah kata berdasarkan pemisahan whitespace;
- nilai minimum;
- median;
- rata-rata;
- maksimum.

Fitur tersebut hanya digunakan untuk **profiling**. Transformasi cleaning akan dilakukan pada Milestone 2.


In [19]:
text_candidates = [
    column
    for column in ["Sentence_Unit", "Title"]
    if column in schema and schema[column] == pl.String
]

print("Kolom teks yang ditemukan:", text_candidates)


Kolom teks yang ditemukan: ['Sentence_Unit', 'Title']


In [20]:
for column in text_candidates:
    text_profile = (
        lf
        .select([
            pl.col(column).str.len_chars().alias("character_length"),
            pl.col(column).str.split(" ").list.len().alias("word_count"),
        ])
        .select([
            pl.col("character_length").count().alias("non_null_count"),
            pl.col("character_length").min().alias("min_characters"),
            pl.col("character_length").median().alias("median_characters"),
            pl.col("character_length").mean().alias("mean_characters"),
            pl.col("character_length").max().alias("max_characters"),
            pl.col("word_count").min().alias("min_words"),
            pl.col("word_count").median().alias("median_words"),
            pl.col("word_count").mean().alias("mean_words"),
            pl.col("word_count").max().alias("max_words"),
        ])
        .collect()
    )

    print(f"\n=== Profil teks: {column} ===")
    display(text_profile)



=== Profil teks: Sentence_Unit ===


non_null_count,min_characters,median_characters,mean_characters,max_characters,min_words,median_words,mean_words,max_words
u32,u32,f64,f64,u32,u32,f64,f64,u32
3013172,25,108.0,122.478431,500,5,15.0,17.083286,93



=== Profil teks: Title ===


non_null_count,min_characters,median_characters,mean_characters,max_characters,min_words,median_words,mean_words,max_words
u32,u32,f64,f64,u32,u32,f64,f64,u32
0,null,null,null,null,null,null,null,null


## 14. Pemeriksaan Awal Kolom Date dan Location

Apabila dataset memiliki kolom `Date` atau `Location`, informasi awal mengenai kelengkapan dan keragaman nilainya diperiksa.

Pada tahap ini belum dilakukan analisis temporal atau spasial. Analisis tersebut menjadi bagian dari **Milestone 3**.


In [21]:
for column in ["Date", "Location"]:
    if column in schema:
        profile = (
            lf
            .select([
                pl.col(column).null_count().alias("missing"),
                pl.col(column).n_unique().alias("unique_values"),
            ])
            .collect()
        )

        print(f"\n=== {column} ===")
        display(profile)
    else:
        print(f"Kolom {column} tidak ditemukan.")



=== Date ===


missing,unique_values
u32,u32
3013172,1



=== Location ===


missing,unique_values
u32,u32
0,9


## 15. Ringkasan Profiling

Bagian ini merangkum kondisi awal dataset berdasarkan hasil pemeriksaan sebelumnya.

Ringkasan ini sengaja hanya berisi **temuan profiling**, bukan keputusan cleaning. Keputusan mengenai missing values, duplikasi, outlier, dan aturan kualitas akan dilakukan pada Milestone 2.


In [22]:
print("=" * 70)
print("RINGKASAN DATA PROFILING")
print("=" * 70)

print(f"Dataset          : {DATA_PATH.name}")
print(f"Ukuran file      : {file_size_mb:,.2f} MB")
print(f"Jumlah baris     : {row_count:,}")
print(f"Jumlah kolom     : {column_count:,}")
print(f"Jumlah duplikat  : {duplicate_count:,}")
print(f"Kolom kategorikal: {len(categorical_columns):,}")
print(f"Kolom numerik    : {len(numeric_columns):,}")
print(f"Kolom teks       : {len(text_candidates):,}")

print("\nSyarat Milestone 1:")
print(f"- >= 500 MB      : {'YA' if meets_size_requirement else 'TIDAK'}")
print(f"- > 1 juta baris : {'YA' if meets_row_requirement else 'TIDAK'}")
print(
    f"- Status         : "
    f"{'TERPENUHI' if (meets_size_requirement or meets_row_requirement) else 'BELUM TERPENUHI'}"
)


RINGKASAN DATA PROFILING
Dataset          : data_indonesian_religius.csv
Ukuran file      : 682.73 MB
Jumlah baris     : 3,013,172
Jumlah kolom     : 7
Jumlah duplikat  : 355
Kolom kategorikal: 7
Kolom numerik    : 0
Kolom teks       : 2

Syarat Milestone 1:
- >= 500 MB      : YA
- > 1 juta baris : YA
- Status         : TERPENUHI


## 16. Kesimpulan Awal

Berdasarkan profiling yang telah dilakukan, dataset telah diperiksa dari sisi ukuran file, jumlah baris dan kolom, tipe data, missing values, duplikasi, statistik dasar, distribusi kategorikal, karakteristik teks, serta keberadaan informasi tanggal dan lokasi.

### Kesimpulan yang perlu dicatat setelah notebook dijalankan

1. Dataset memiliki ukuran dan/atau jumlah baris yang memenuhi persyaratan skala Milestone 1.
2. Struktur kolom dan tipe data telah teridentifikasi.
3. Missing values dan duplikasi telah dipetakan tanpa melakukan perubahan terhadap data asli.
4. Karakteristik variabel teks telah diprofilkan karena dataset merupakan corpus teks.
5. Hasil profiling menjadi dasar untuk menentukan strategi cleaning pada **`02_data_cleaning.ipynb`**.

> **Catatan:** Jangan mengubah angka kesimpulan secara manual. Gunakan angka yang dihasilkan oleh cell profiling setelah notebook dijalankan pada dataset lokal.


---

### Status Milestone 1

Checklist:

- [x] Dataset Indonesia tersedia di `data/raw/`
- [x] Dataset memenuhi syarat ukuran atau jumlah baris
- [x] `data/README.md` mendokumentasikan dataset
- [x] `01_data_profiling.ipynb` melakukan profiling awal
- [x] Profiling menggunakan Polars
- [x] Lazy API `scan_csv()` digunakan
- [ ] Cleaning data — **Milestone 2**
- [ ] DuckDB `SUMMARIZE` — **Milestone 2**
- [ ] Minimal 6 visualisasi interaktif — **Milestone 3**
- [ ] Minimal 5 insight analitik — **Final**
